# Notebook: exportar un checkpoint YOLOX `.pth` a ONNX con clases empaquetadas

Este notebook esta pensado para usuarios que ya tienen un modelo YOLOX entrenado en formato `.pth` y solo necesitan generar el archivo ONNX final con los nombres de clase embebidos en el metadato `names`.


## Que hace este notebook

Este notebook hace solo la parte de exportacion. No entrena el modelo y no necesita dataset.

Flujo:

1. instala YOLOX y dependencias,
2. recibe tu archivo `.pth`,
3. te pide ingresar manualmente las clases,
4. reconstruye la configuracion del modelo,
5. exporta a ONNX,
6. inserta las clases en el metadato `names`,
7. descarga el ONNX final.

Importante:

- Debes indicar la misma variante YOLOX con la que entrenaste el `.pth`.
- Debes ingresar las clases en el mismo orden exacto usado durante el entrenamiento.
- Si la variante o el orden de clases no coinciden, la exportacion puede fallar o el ONNX puede quedar mal configurado.


## Como ingresar las clases

En este notebook las clases se escriben manualmente en una lista de Python.

Ejemplo:

```python
CLASS_NAMES = [
    "bicycle",
    "bus",
    "car",
    "motorcycle",
    "truck",
]
```

Reglas:

- No agregues clases de mas ni de menos.
- El orden debe coincidir exactamente con el entrenamiento.
- Si durante el entrenamiento la clase 0 era `bicycle`, aqui tambien debe estar en la posicion 0.


In [ ]:
# ======================================
# 1) CONFIGURACION GENERAL DEL NOTEBOOK
# ======================================
from pathlib import Path

BASE_DIR = Path("/content")
YOLOX_DIR = BASE_DIR / "YOLOX"
EXP_FILE = YOLOX_DIR / "yolox_export_config.py"

# Debe coincidir con la variante usada en el entrenamiento del .pth
YOLOX_VARIANT = "yolox_s"   # yolox_nano, yolox_tiny, yolox_s, yolox_m, yolox_l, yolox_x

# Tamano de entrada usado por el modelo. Normalmente 640.
INPUT_SIZE = 640

# Escribe las clases en el mismo orden exacto usado durante el entrenamiento.
CLASS_NAMES = [
    "bicycle",
    "bus",
    "car",
    "motorcycle",
    "truck",
]

MODEL_SCALES = {
    "yolox_nano": (0.33, 0.25),
    "yolox_tiny": (0.33, 0.375),
    "yolox_s":    (0.33, 0.50),
    "yolox_m":    (0.67, 0.75),
    "yolox_l":    (1.00, 1.00),
    "yolox_x":    (1.33, 1.25),
}

assert YOLOX_VARIANT in MODEL_SCALES, f"Variante no soportada: {YOLOX_VARIANT}"
if not CLASS_NAMES:
    raise ValueError("Debes ingresar al menos una clase en CLASS_NAMES")
if len(set(CLASS_NAMES)) != len(CLASS_NAMES):
    raise ValueError("CLASS_NAMES contiene nombres repetidos. Revísalo antes de continuar.")

NUM_CLASSES = len(CLASS_NAMES)
depth, width = MODEL_SCALES[YOLOX_VARIANT]

print("Configuracion actual")
print("YOLOX_VARIANT =", YOLOX_VARIANT)
print("INPUT_SIZE    =", INPUT_SIZE)
print("NUM_CLASSES   =", NUM_CLASSES)
print("CLASS_NAMES   =", CLASS_NAMES)
print("depth,width   =", (depth, width))


In [ ]:
# =================================
# 2) INSTALAR DEPENDENCIAS Y YOLOX
# =================================
%cd /content

import re
import shutil

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q pycocotools onnx onnxscript onnxruntime loguru tabulate thop ninja "jedi>=0.16"

if YOLOX_DIR.exists():
    shutil.rmtree(YOLOX_DIR)
    print("YOLOX anterior eliminado")

!git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

req_file = YOLOX_DIR / "requirements.txt"
txt = req_file.read_text(encoding="utf-8")
txt = re.sub(r'^.*onnx-simplifier.*\n?', '', txt, flags=re.MULTILINE)
txt = re.sub(r'^.*onnxsim.*\n?', '', txt, flags=re.MULTILINE)
req_file.write_text(txt, encoding="utf-8")

!python -m pip install -q -r /content/YOLOX/requirements.txt

print("Instalacion terminada.")


In [ ]:
# ======================================================
# 3) FORZAR PYTHONPATH Y VALIDAR IMPORTACION DE YOLOX
# ======================================================
import os
import sys

if str(YOLOX_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOX_DIR))

current_pythonpath = os.environ.get("PYTHONPATH", "")
paths = [p for p in current_pythonpath.split(":") if p] if current_pythonpath else []
if str(YOLOX_DIR) not in paths:
    os.environ["PYTHONPATH"] = f"{YOLOX_DIR}:{current_pythonpath}" if current_pythonpath else str(YOLOX_DIR)

import yolox
print("PYTHONPATH =", os.environ.get("PYTHONPATH"))
print("Importacion OK de yolox desde:", yolox.__file__)


In [ ]:
# =========================
# 4) SUBIR CHECKPOINT .PTH
# =========================
from google.colab import files

print("Sube un unico archivo .pth entrenado con YOLOX.")
uploaded = files.upload()

pth_candidates = [name for name in uploaded.keys() if name.lower().endswith(".pth")]
if not pth_candidates:
    raise FileNotFoundError("No se subio ningun archivo .pth")
if len(pth_candidates) > 1:
    raise ValueError("Se detectaron varios .pth. Sube solo un checkpoint por ejecucion.")

CHECKPOINT_PATH = BASE_DIR / pth_candidates[0]
print("Checkpoint detectado:", CHECKPOINT_PATH)


In [ ]:
# ==============================================
# 5) CREAR CONFIGURACION YOLOX PARA EXPORTACION
# ==============================================
config_text = f'''
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.num_classes = {NUM_CLASSES}
        self.depth = {depth}
        self.width = {width}
        self.input_size = ({INPUT_SIZE}, {INPUT_SIZE})
        self.test_size = ({INPUT_SIZE}, {INPUT_SIZE})
        self.random_size = ({INPUT_SIZE // 32}, {INPUT_SIZE // 32})
        self.exp_name = "yolox_export_manual_classes"

def get_exp():
    return Exp()
'''

EXP_FILE.write_text(config_text, encoding="utf-8")
print("Archivo creado:", EXP_FILE)
print(EXP_FILE.read_text(encoding="utf-8"))


In [ ]:
# =========================================================
# 6) PARCHEAR tools/export_onnx.py PARA COLAB ACTUAL
# =========================================================
export_file = YOLOX_DIR / "tools" / "export_onnx.py"
txt = export_file.read_text(encoding="utf-8")

txt = txt.replace(
    'ckpt = torch.load(ckpt_file, map_location="cpu")',
    'ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)'
)

txt = txt.replace(
    "torch.onnx._export(",
    "torch.onnx.export("
)

export_file.write_text(txt, encoding="utf-8")
print("Parche aplicado en:", export_file)


In [ ]:
# =========================================
# 7) EXPORTAR EL CHECKPOINT A ONNX
# =========================================
%cd /content/YOLOX

import subprocess

output_dir = BASE_DIR / "onnx_export_output"
output_dir.mkdir(parents=True, exist_ok=True)
onnx_path = output_dir / "modelo_yolox_exportado.onnx"

cmd = [
    "python", "tools/export_onnx.py",
    "-f", str(EXP_FILE),
    "-c", str(CHECKPOINT_PATH),
    "--output-name", str(onnx_path),
    "--no-onnxsim",
    "--decode_in_inference",
]

env = os.environ.copy()
env["PYTHONPATH"] = "/content/YOLOX:" + env.get("PYTHONPATH", "")

print("Comando real:")
print("PYTHONPATH=" + env["PYTHONPATH"])
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd="/content/YOLOX",
    env=env,
    capture_output=True,
    text=True
)

print("\n===== STDOUT =====")
print(result.stdout if result.stdout else "(vacio)")

print("\n===== STDERR =====")
print(result.stderr if result.stderr else "(vacio)")

print("\nCodigo de salida:", result.returncode)
print("Existe el ONNX?:", onnx_path.exists(), onnx_path)

if not onnx_path.exists():
    raise FileNotFoundError(f"No se genero el ONNX en {onnx_path}")

print("Modelo ONNX generado en:", onnx_path)


In [ ]:
# ===================================================
# 8) INSERTAR LAS CLASES EN EL METADATO DEL ONNX
# ===================================================
import json
import onnx
import uuid

random_id = uuid.uuid4().hex[:8]
onnx_out = output_dir / f"modelo_yolox_{random_id}.onnx"
class_dict = {idx: name for idx, name in enumerate(CLASS_NAMES)}

model = onnx.load(str(onnx_path))
kept = [p for p in model.metadata_props if p.key != "names"]
del model.metadata_props[:]
model.metadata_props.extend(kept)

model.metadata_props.append(
    onnx.StringStringEntryProto(
        key="names",
        value=json.dumps(class_dict, ensure_ascii=False)
    )
)

onnx.save(model, str(onnx_out))
print("ONNX final con metadato 'names' guardado en:", onnx_out)
print("Las clases embebidas en el modelo son:", class_dict)


In [ ]:
# ============================================
# 9) VERIFICAR METADATO DEL ONNX GENERADO
# ============================================
m = onnx.load(str(onnx_out))
meta = {p.key: p.value for p in m.metadata_props}

print("Metadata encontrada:")
for k, v in meta.items():
    print(f"{k} = {v}")

if "names" not in meta:
    raise KeyError("No se encontro el metadato 'names' en el ONNX final")

names = json.loads(meta["names"])
print("\nClases recuperadas desde metadata:")
print(names)


## Nota final

El archivo descargado es el ONNX final con el metadato `names`. Ese es el archivo que debes conservar para tu integracion.

Si en el futuro cambias las clases o el orden de las clases, tendras que volver a exportar y volver a empaquetar el ONNX.


In [ ]:
# ==================================
# 10) DESCARGAR SOLO EL ONNX FINAL
# ==================================
from google.colab import files

if not onnx_out.exists():
    raise FileNotFoundError(f"No existe el archivo final esperado: {onnx_out}")

print("Descargando:", onnx_out)
print("Este es el archivo final que debes conservar y usar fuera de Colab.")
files.download(str(onnx_out))
